In [2]:
import os
import json
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 配置参数
population_file = 'data/output/population_points.json'  # 你可以修改为你实际的文件路径

# 检查文件是否存在
if not os.path.exists(population_file):
    print(f"❌ 找不到人口数据文件: {population_file}")
else:
    try:
        # 读取JSON文件
        with open(population_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # 提取学生代理数据
        agents = data['agents']['student_agent']['states']['default']['agents']
        num_student_agents = len(agents)
        print(f"找到 {num_student_agents} 个学生代理个体")
        
        
    except Exception as e:
        print(f"❌ 处理文件时出错: {str(e)}")

找到 1914 个学生代理个体


In [ ]:
agent_data=agents[3]['variables']
agent_data

In [ ]:
agent_data['x'] 


256.53206457581945

In [1]:
from pyflamegpu import *
import pyflamegpu.codegen
import sys

In [6]:
?SimulationConfig()

Object `SimulationConfig()` not found.


In [1]:
AGENT_COUNT = 16384
ENV_WIDTH = int(AGENT_COUNT**(1/3))

In [3]:
stairwell_file = 'data/output/transformed_stairwell.geojson'

with open(stairwell_file, 'r', encoding='utf-8') as f:
    stairwell_data = json.load(f)

# 提取楼梯间特征
stairwell_features = stairwell_data['features']
num_stairwell_agents = len(stairwell_features)
print(f"初始化 {num_stairwell_agents} 个楼梯间代理")
    


初始化 7 个楼梯间代理


In [6]:
feature = stairwell_features[1]
coordinates = feature['geometry']['coordinates']
coordinates[0]

320.31274575542193

In [3]:
config_path = os.path.join(os.path.dirname(__file__), '../config/env.yaml')
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    base_population = config.get('base_population', 50)

NameError: name '__file__' is not defined

In [2]:
from pyflamegpu import *

pyflamegpu.clearRTCDiskCache()

In [14]:
import numpy as np
import json
from shapely.geometry import Point

def generate_attraction_matrix(
    m, n,
    attraction_points,  # 手动输入的高吸引力点列表 [(x, y, radius, attraction), ...]
    normalize=False,     # 是否归一化到 [0, 1]
    output_json=False    # 是否返回 JSON 格式
):
    """
    Generate an attraction matrix based on specified attraction points with radial influence.
    
    Parameters:
    - m, n: Dimensions of the output matrix (rows, columns)
    - attraction_points: List of tuples (x, y, radius, attraction_value)
    - normalize: Whether to normalize the matrix to [0, 1]
    - output_json: Whether to return the result as JSON
    
    Returns:
    - Either a numpy array or JSON string representing the attraction matrix
    """
    
    # 1. 初始化总体边界矩阵
    attraction_matrix = np.zeros((m, n))
    
    # 2. 计算每个网格点的吸引力
    for x, y, radius, attraction in attraction_points:
        # 确保坐标在矩阵范围内
        x = max(0, min(m-1, x))
        y = max(0, min(n-1, y))
        
        # 创建网格坐标
        rows, cols = np.indices((m, n))
        
        # 计算每个点到吸引力中心的距离
        distances = np.sqrt((rows - x)**2 + (cols - y)**2)
        
        # 应用高斯衰减函数 (在半径范围内)
        decay = np.exp(-(distances**2) / (2 * (radius/3)**2))  # radius/3 makes it fall to ~0.1 at radius
        influence = attraction * np.where(distances <= radius, decay, 0)
        
        # 叠加到总体矩阵
        attraction_matrix += influence
    
    # 3. 归一化（可选）
    if normalize:
        max_val = np.max(attraction_matrix)
        if max_val > 0:
            attraction_matrix = attraction_matrix / max_val
    
    # 4. 返回 JSON 或矩阵
    if output_json:
        json_data = {
            "macro_environment": {
                "map": attraction_matrix.flatten().tolist()  # 展平为 1D 数组
            }
        }
        return json.dumps(json_data, indent=2)
    else:
        return attraction_matrix

In [15]:
m, n = 20,20  # 5x5 网格

# 手动输入高吸引力点 (x, y, radius, attraction)
attraction_points = [
    (10, 10, 4, 100),  # 中心点 (2,2)，半径 2，吸引力 100
    (0, 4, 4, 80),  # 点 (0,4)，半径 1.5，吸引力 80
]

# 生成 JSON 格式
output = generate_attraction_matrix(m, n, attraction_points)

output

array([[  0.88871972,   6.3647607 ,  25.97219739,  60.38716816,
         80.        ,  60.38716816,  25.97219739,   6.3647607 ,
          0.88871972,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   4.80437343,  19.60484314,  45.58262598,
         60.38716816,  45.58262598,  19.60484314,   4.80437343,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   2.06633526,   8.43193796,  19.60484314,
         25.97219739,  19.60484314,   8.43193796,   2.06633526,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   0.        ,  

In [12]:
import numpy as np
from shapely.geometry import Point

def generate_attraction_matrix(
    m, n,
    attraction_points,  # 手动输入的高吸引力点列表 [(x, y, radius, attraction), ...]
    normalize=False,     # 是否归一化到 [0, 1]
    output_json=False   # 是否返回 JSON 格式
):
    """
    生成一个 m×n 的吸引力矩阵，基础值为1，高吸引力点在其周边衰减叠加。
    
    参数:
        m (int): 网格行数
        n (int): 网格列数
        attraction_points (list): 高吸引力点列表，格式 [(x, y, radius, attraction), ...]
        normalize (bool): 是否归一化到 [0, 1]（默认 True）
        output_json (bool): 是否返回 JSON 格式（默认 False）
    
    返回:
        dict: 如果 output_json=True，返回 JSON 结构
        np.ndarray: 如果 output_json=False，返回吸引力矩阵
    """
    # 1. 初始化吸引力矩阵（基础值 = 1）
    attraction_matrix = np.ones((m, n))
    
    # 2. 计算高吸引力点的影响（叠加到基础值上）
    for i in range(m):
        for j in range(n):
            point = Point(j + 0.5, i + 0.5)  # 单元格中心点坐标
            
            # 计算所有高吸引力点的影响
            for x, y, radius, attraction in attraction_points:
                dist = np.sqrt((point.x - x) ** 2 + (point.y - y) ** 2)
                if dist <= radius:
                    # 高斯衰减
                    gaussian_factor = np.exp(-(dist ** 2) / (2 * (radius / 2) ** 2))
                    attraction_matrix[i, j] += attraction * gaussian_factor
    
    # 3. 归一化（可选）
    if normalize:
        max_val = np.max(attraction_matrix)
        if max_val > 0:
            attraction_matrix = attraction_matrix / max_val
    
    # 4. 返回 JSON 或矩阵
    if output_json:
        json_data = {
            "macro_environment": {
                "map": attraction_matrix.flatten().tolist()  # 展平为 1D 数组
            }
        }
        return json_data
    else:
        return attraction_matrix



In [13]:

m, n = 20, 18  # 5x5 网格

# 手动输入高吸引力点 (x, y, radius, attraction)
attraction_points = [
    (10, 10, 2, 5),  # 中心点 (2,2)，半径 2，额外吸引力 5
    (0, 4, 1.5, 3), # 点 (0,4)，半径 1.5，额外吸引力 3
]

# 生成矩阵
matrix = generate_attraction_matrix(m, n, attraction_points, normalize=True)

matrix

array([[0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167],
       [0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167],
       [0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167],
       [0.59737205, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167],
    

In [23]:
def generate_attraction_matrix(
    m, n,
    attraction_points,  # 手动输入的高吸引力点列表 [(x, y, radius, attraction), ...]
    normalize=False,     # 是否归一化到 [0, 1]
    output_json=False    # 是否返回 JSON 格式
):
    """
    Generate an attraction matrix based on specified attraction points with radial influence.
    
    Parameters:
    - m, n: Dimensions of the output matrix (rows, columns)
    - attraction_points: List of tuples (x, y, radius, attraction_value)
    - normalize: Whether to normalize the matrix to [0, 1]
    - output_json: Whether to return the result as JSON
    
    Returns:
    - Either a numpy array or JSON string representing the attraction matrix
    """
    
    # 1. 初始化总体边界矩阵
    attraction_matrix = np.zeros((m, n))
    
    # 2. 计算每个网格点的吸引力
    for x, y, radius, attraction in attraction_points:
        # 确保坐标在矩阵范围内
        x = max(0, min(m-1, x))
        y = max(0, min(n-1, y))
        
        # 创建网格坐标
        rows, cols = np.indices((m, n))
        
        # 计算每个点到吸引力中心的距离
        distances = np.sqrt((rows - x)**2 + (cols - y)**2)
        
        # 应用高斯衰减函数 (在半径范围内)
        decay = np.exp(-(distances**2) / (2 * (radius/2)**2))  # radius/3 makes it fall to ~0.1 at radius
        influence = attraction * np.where(distances <= radius, decay, 0)
        
        # 叠加到总体矩阵
        attraction_matrix += influence
    
    # 3. 归一化（可选）
    if normalize:
        max_val = np.max(attraction_matrix)
        if max_val > 0:
            attraction_matrix = attraction_matrix / max_val
    
    # 4. 返回 JSON 或矩阵
    if output_json:
        json_data = {
            "macro_environment": {
                "map": attraction_matrix.flatten().tolist()  # 展平为 1D 数组
            }
        }
        return json.dumps(json_data, indent=2)
    else:
        return attraction_matrix

In [24]:
m, n = 20, 18  # 5x5 网格

# 手动输入高吸引力点 (x, y, radius, attraction)
attraction_points = [
    (10, 10, 2, 100),  # 中心点 (2,2)，半径 2，额外吸引力 5
    (0, 4, 1.5, 80), # 点 (0,4)，半径 1.5，额外吸引力 3
]

# 生成矩阵
matrix = generate_attraction_matrix(m, n, attraction_points, normalize=False)

matrix

array([[  0.        ,   0.        ,   0.        ,  32.88898324,
         80.        ,  32.88898324,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,  13.52106523,
         32.88898324,  13.52106523,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.   

第二种处理方法：

In [33]:
import numpy as np
from collections import deque

def generate_attraction_matrix_direct(
    m, n,
    attraction_points,  # [(x, y, radius, base_attraction)]
    decay_rate=0.5,     # 每层衰减比例 (0~1)
    normalize=False,
    output_json=True
):
    """
    通过广度优先搜索 (BFS) 实现逐层衰减的吸引力矩阵
    """
    attraction_matrix = np.zeros((m, n))
    
    for x, y, radius, base_attraction in attraction_points:
        x, y = int(np.clip(x, 0, m-1)), int(np.clip(y, 0, n-1))
        visited = np.zeros((m, n), dtype=bool)
        queue = deque()
        
        # 初始化中心点
        queue.append((x, y, base_attraction))
        visited[x, y] = True
        attraction_matrix[x, y] += base_attraction
        
        # BFS 向外扩散
        while queue:
            cx, cy, current_attraction = queue.popleft()
            
            # 遍历四个邻居方向
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nx, ny = cx + dx, cy + dy
                
                # 检查边界和是否已访问
                if 0 <= nx < m and 0 <= ny < n and not visited[nx, ny]:
                    distance = abs(nx - x) + abs(ny - y)  # 曼哈顿距离
                    if distance <= radius:
                        decayed_attraction = base_attraction * (decay_rate ** distance)
                        attraction_matrix[nx, ny] += decayed_attraction
                        visited[nx, ny] = True
                        queue.append((nx, ny, decayed_attraction))
    
    if normalize:
        max_val = np.max(attraction_matrix)
        if max_val > 0:
            attraction_matrix /= max_val
    
    if output_json:
        return json.dumps({"macro_environment": {"map": attraction_matrix.flatten().tolist()}})
    else:
        return attraction_matrix

In [35]:
m, n = 20, 18  # 5x5 网格

# 手动输入高吸引力点 (x, y, radius, attraction)
attraction_points = [
    (10, 10, 2, 100),  # 中心点 (2,2)，半径 2，额外吸引力 5
    (0, 4, 2, 80), # 点 (0,4)，半径 1.5，额外吸引力 3
]

# 生成矩阵
matrix = generate_attraction_matrix_direct(m, n, attraction_points, normalize=False)

